# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates a step-by-step workflow for accessing, loading, and exploring the FAIR² (FAIR2) dataset using the `mlcroissant` library. All entities—record sets, fields, and columns—are referenced by their unique `@id` values.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Install `mlcroissant` if not already available
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

All references to dataset entities will use their unique `@id` fields, as specified by the Croissant schema.

In [ ]:
# List all record sets and details
print("Available Record Sets:")
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in this dataset metadata.")
else:
    for rs in record_sets:
        print(f"- Record Set name: {getattr(rs, 'name', '<unknown>')}, @id: {rs.id}")
        # List all fields in the record set
        if hasattr(rs, 'fields'):
            fields = rs.fields
            for f in fields:
                print(f"    - Field: {getattr(f, 'name', '<unknown>')} (@id: {f.id}) — Type: {getattr(f, 'data_type', 'N/A')}")
                # If columns exist on field, list them
                if hasattr(f, 'columns'):
                    for col in f.columns:
                        print(f"      - Column: {getattr(col, 'name', '<unknown>')} (@id: {col.id})")

## 3. Data Extraction
Now we will attempt to load all available record sets and extract their contents as DataFrames, making references strictly by their `@id` fields, as mandated by the Croissant schema.

If the dataset contains no record sets, the code will proceed with an empty dictionary and print an informative message.

In [ ]:
# Extract all available record sets, loading into a dictionary by record set @id
dataframes = {}
record_set_ids = []
if not dataset.record_sets:
    print("No record sets defined in this dataset; nothing to extract.")
else:
    # Use only @id reference for each record set
    for rs in dataset.record_sets:
        rs_id = rs.id
        record_set_ids.append(rs_id)
        print(f"Loading records for record set with @id: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
    print("Loaded record sets:", record_set_ids)
    # Show columns for the first record set
    example_rs = record_set_ids[0] if record_set_ids else None
    if example_rs:
        print("Columns / Fields (@id):", dataframes[example_rs].columns.tolist())
        display(dataframes[example_rs].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply common steps: filtering rows by a numeric field, normalization, and grouping.

You **must** replace `<numeric_field_id>`, `<record_set_id>`, and `<group_field>` below with the `@id` values found above for your exploration.

The code will use placeholders if no real record sets are available.

In [ ]:
# Only proceed if at least one record set and at least one numeric field exists
if not dataframes:
    print("No record sets loaded; unable to perform EDA.")
else:
    # For demonstration, use the first available record set and numeric field if available
    selected_rs = record_set_ids[0]
    df = dataframes[selected_rs]

    print(f"Using record set: {selected_rs}")
    
    # Find a likely numeric field by inferring types
    numeric_field = None
    for col in df.columns:
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
        except Exception:
            continue

    if numeric_field is None:
        print("No numeric field found in the record set; skipping EDA.")
    else:
        print(f"Numeric field chosen: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() else 1)
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt grouping by a non-numeric field
        group_field = None
        for col in df.columns:
            if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")

## 5. Visualization
Let's visualize the distribution of a numeric field, if available. This will help us understand its nature and potential for further modeling or decision making.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None:
    plt.figure(figsize=(7, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
    plt.title(f'Distribution of {numeric_field} in record set {selected_rs}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
else:
    print("Cannot plot: Missing numeric field or data.")

## 6. Conclusion
In this notebook, we used `mlcroissant` to load and explore the FAIR² dataset defined via a Croissant schema. All references to record sets, fields, and columns are by their `@id`s as required. This process enables:
- Loading official and reproducible metadata and data from Croissant-compliant datasets
- Discovering and referencing schema components by their unique identifiers
- Performing standard EDA and preparing data for machine learning and reporting

**Note**: The precise available record sets and fields may differ or may be absent depending on the dataset's Croissant schema completeness.